# \# Decision Tree for Classifier

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
titanic = sns.load_dataset("titanic")

In [ ]:
titanic.head()
# titanic.info()

In [ ]:
features = ["pclass", "sex", "fare","embarked","age"]
target = ["survived"]

In [ ]:
# fill null values
from sklearn.impute import SimpleImputer

imp_median = SimpleImputer(strategy="median") 
titanic[["age"]] = imp_median.fit_transform(titanic[["age"]])

imp_freq = SimpleImputer(strategy="most_frequent")
titanic[["embarked"]] = imp_freq.fit_transform(titanic[["embarked"]])


In [ ]:
# encoding
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
titanic["sex"] = le.fit_transform(titanic["sex"])
titanic["embarked"] = le.fit_transform(titanic["embarked"])


In [ ]:
titanic.head()

In [ ]:
X = titanic[features]
y = titanic[target]

In [ ]:
X.head()
y.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3,random_state=42)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier()
model.fit(X_train, y_train)


In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score
print("accuracy score : ", accuracy_score(y_test, y_pred))

In [ ]:
from sklearn.tree import plot_tree

# plt.figure(figsize=(18,10))

# plot_tree(model, feature_names = X.columns, class_names=["Died","Survived"], filled=True, max_depth=2)
# plt.title("Decision Tree")
# plt.tight_layout()
# plt.show()


# \# Decision Tree with pre pruning

In [ ]:
max_depth = [2,3,4,5,6,7,8,9,10]

for depth in max_depth:
    model = DecisionTreeClassifier(max_depth = depth)
    model.fit(X_train, y_train)

    acc = model.score(X_test, y_test)
    print(f"Accuracy in depth {depth}: ", acc)

    # if(depth==4):
    #     plt.figure(figsize=(18,10))

    #     plot_tree(model, feature_names = X.columns, class_names=["Died","Survived"], filled=True)
    #     plt.title("Decision Tree")
    #     plt.tight_layout()
    #     plt.show()

In [ ]:
min_samples_split = [5,10, 15, 20, 25, 30]

for split in min_samples_split:
    model = DecisionTreeClassifier(max_depth = 4, min_samples_split=split)
    model.fit(X_train, y_train)

    acc = model.score(X_test, y_test)
    print(f"Sample split at split {split}: ", acc)

    # if(split == 10):
    #     plt.figure(figsize=(18,10))

    #     plot_tree(model, feature_names = X.columns, class_names=["Died","Survived"], filled=True)
    #     plt.title("Decision Tree")
    #     plt.tight_layout()
    #     plt.show()

# \# Decision Tree with post-pruning

In [ ]:
full_tree = DecisionTreeClassifier(random_state=42)
full_tree.fit(X_train,y_train)

In [ ]:
path = full_tree.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas
ccp_alphas

In [ ]:
# train our model for all alphas

trees = []

for alpha in ccp_alphas:
    model = DecisionTreeClassifier(random_state=42, ccp_alpha = alpha)
    model.fit(X_train, y_train)

    trees.append([model, alpha])

In [ ]:
best_acc = 0
best_alpha = 0

for model, alpha in trees:
    curr_acc = model.score(X_test, y_test)
    if(curr_acc > best_acc):
        best_acc = curr_acc
        best_alpha = alpha


In [ ]:
print("best alpha : ", best_alpha)
print("best accuracy : ", best_acc)


In [ ]:
best_model = DecisionTreeClassifier(ccp_alpha=best_alpha)
best_model.fit(X_train, y_train)

In [ ]:
plt.figure(figsize=(18,10))

plot_tree(best_model, feature_names = X.columns, class_names=["Died","Survived"], filled=True)
plt.title("Decision Tree")
plt.tight_layout()
plt.show()

In [ ]:
print("best score : ", best_model.score(X_test, y_test))